### Pydantic

In [2]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002AF7985D6A0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002AF7985E120>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [12]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    ratings:float=Field(description="The movies rating out of 10")


In [13]:
pydantic_model = model.with_structured_output(Movie)
pydantic_model

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000225D8A392B0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000225D8A39D30>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'ratings': {'description': 'The movies rating ou

In [5]:
model.invoke("Provide details about the movie Inception")

AIMessage(content='<think>\nOkay, I need to provide details about the movie Inception. Let me start by recalling what I know about it. Directed by Christopher Nolan, it\'s a sci-fi action film that came out in 2010. The main actor is Leonardo DiCaprio, right? The title "Inception" probably relates to entering someone\'s mind or planting an idea. \n\nSo, the plot must involve something about dreams and controlling them. I remember hearing that it\'s a complex narrative with layers of dreams within dreams. The concept is probably inspired by the idea of entering a person\'s subconscious to implant or extract an idea. That\'s a common theme in science fiction, but Nolan usually adds his unique twist.\n\nThe main character\'s name is Dom Cobb, played by DiCaprio. He\'s a thief who steals information by infiltrating the subconscious. But he\'s offered a chance to erase his criminal past by performing the reverse, which is called "inception"—planting an idea instead. That\'s the central conf

In [14]:
pydantic_model.invoke("Provide details about the movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', ratings=8.8)

## Message output alongside parsed structure

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details"""
    title:str=Field(..., description="The title of the movie")
    year:int=Field(..., description="This year the movie was released")
    director:str=Field(..., description="The director of the movie")
    ratings:float=Field(..., description="The movies rating out of 10")

pydantic_model = model.with_structured_output(Movie, include_raw=True) ## Include the raw message too from the LLM


In [16]:
pydantic_model.invoke("Provide details about the movie Inception")

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me see what I need to do. The available tool is the Movie function, which requires title, year, director, and ratings. I need to provide those parameters. \n\nFirst, I know the title is "Inception". The year it was released was 2010. The director is Christopher Nolan. As for ratings, I think it\'s around 8.8 on IMDb. Let me double-check that. Yes, IMDb gives it 8.8/10. So I should structure the tool call with these details. Make sure all required fields are included. No need to add extra info. Just the parameters specified in the function. Alright, that should cover it.\n', 'tool_calls': [{'id': '6dbpca7tf', 'function': {'arguments': '{"director":"Christopher Nolan","ratings":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 203, 'prompt_tokens': 230, 'total_tokens

### Nested Structure

In [17]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

pydantic_model = model.with_structured_output(MovieDetails)

response = pydantic_model.invoke("Provide details about the moview Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne')], genres=['Science Fiction', 'Action', 'Thriller'], budget=160.0)

### TypedDict

In [3]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title:Annotated[str, ..., "The title of the movie"]
    director:Annotated[str, ..., "The director of the movie"]
    rating:Annotated[float, ..., "The movie's rating out of 10"]

typedict_model = model.with_structured_output(MovieDict)
response = typedict_model.invoke("Please provide details for the movie Avengers")

response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers'}

In [5]:
from pydantic import Field

class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

typedict_model = model.with_structured_output(MovieDetails)
response = typedict_model.invoke("Please provide details for the movie Avengers")

response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Iron Man'},
  {'name': 'Chris Evans', 'role': 'Captain America'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Mark Ruffalo', 'role': 'Hulk'},
  {'name': 'Scarlett Johansson', 'role': 'Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Hawkeye'}],
 'genres': ['Action', 'Superhero', 'Science Fiction', 'Adventure'],
 'title': 'The Avengers',
 'year': 2012}

### Data Classes

In [6]:
import os
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [7]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person"""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent =  create_agent(
    model = "gpt-5",
    response_format = ContactInfo
)

result = agent.invoke({
    "messages" : [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555)123-4567"}]
})

print(result)

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555)123-4567', additional_kwargs={}, response_metadata={}, id='19fb42e7-b74c-4965-a07d-5969ee9781ba'), AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555)123-4567"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 931, 'prompt_tokens': 203, 'total_tokens': 1134, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 896, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DlQax9DvTM2Oy2oHT7JhjtYT1gZHA', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e7bf3-86e0-7d11-9546-edf7c4fa59c5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 203, 'output_to

In [8]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555)123-4567')